# Projeto Integrador — Bloco 4
## Pipeline Completo de Processamento de um Frame do Drone

**Bloco 4 — OpenCV**  
**Projeto:** IC-FAPEMA — Detecção de capacetes em motociclistas via drone  
**Objetivo:** Juntar tudo que foi aprendido no Bloco 4 em um pipeline único: carregar, manipular e desenhar detecções

---

Neste projeto você vai simular o processamento completo de um frame real do drone:
1. Carregar uma imagem real e corrigir as cores (Aula 4.1)
2. Recortar a região de interesse e converter para escala de cinza (Aula 4.2)
3. Desenhar as bounding boxes das detecções do YOLO (Aula 4.3)
4. Salvar o resultado final processado

---

In [ ]:
# Importar todas as bibliotecas do Bloco 4
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Bibliotecas carregadas!")

---
## Etapa 1 — Carregar o frame do drone
*(Aula 4.1 — imread + cvtColor)*

Faça o upload de uma foto de motociclista — este será nosso "frame do drone".

In [ ]:
# ── 1. Upload e carregamento ──────────────────────────────────────────────────
print("Escolha uma foto de motociclista...")
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# Carregar com OpenCV (vem em BGR) e converter para RGB
imagem_bgr = cv2.imread(nome_arquivo)
imagem_rgb = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

altura, largura = imagem_rgb.shape[:2]
print(f"\n✅ Frame carregado: {nome_arquivo}")
print(f"Shape: {imagem_rgb.shape} → altura={altura}px, largura={largura}px")

plt.figure(figsize=(9, 6))
plt.imshow(imagem_rgb)
plt.title('Etapa 1 — Frame original do drone (já convertido BGR→RGB)')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Etapa 2 — Pré-processamento: recorte e escala de cinza
*(Aula 4.2 — slicing/crop + cvtColor para grayscale)*

Antes de rodar o modelo, é comum inspecionar regiões específicas e gerar versões em cinza
para análises auxiliares (ex: detecção de bordas).

In [ ]:
# ── 2. Recortar a região central (onde normalmente está o motociclista) ──────
y1 = altura  // 6
y2 = altura  * 5 // 6
x1 = largura // 6
x2 = largura * 5 // 6

roi = imagem_rgb[y1:y2, x1:x2]   # recorte com slicing NumPy

# ── 3. Converter o frame completo para escala de cinza ────────────────────────
imagem_cinza = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2GRAY)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))

axs[0].imshow(imagem_rgb)
axs[0].set_title(f'Original\n{imagem_rgb.shape}')
axs[0].axis('off')

axs[1].imshow(roi)
axs[1].set_title(f'Recorte (ROI)\n{roi.shape}')
axs[1].axis('off')

axs[2].imshow(imagem_cinza, cmap='gray')
axs[2].set_title(f'Escala de cinza\n{imagem_cinza.shape}')
axs[2].axis('off')

fig.suptitle('Etapa 2 — Pré-processamento do frame', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

---
## Etapa 3 — Desenhar as detecções do YOLO
*(Aula 4.3 — rectangle() + putText())*

Simulamos que o YOLO detectou motociclistas neste frame.  
As coordenadas abaixo representam a posição aproximada de cada detecção (ajuste se quiser).

In [ ]:
# ── 4. Detecções simuladas do YOLO (normalizadas, formato de sempre) ─────────
# classe 0 = com capacete | classe 1 = sem capacete
deteccoes = [
    {'classe': 1, 'x_centro': 0.50, 'y_centro': 0.45, 'largura': 0.35, 'altura': 0.50},
]

# ── 5. Copiar o frame original antes de desenhar ──────────────────────────────
imagem_para_desenhar = imagem_bgr.copy()   # trabalhamos em BGR pois cv2.rectangle espera BGR

for det in deteccoes:
    # Converter coordenadas YOLO → pontos pt1 e pt2
    px1 = int((det['x_centro'] - det['largura'] / 2) * largura)
    py1 = int((det['y_centro'] - det['altura']  / 2) * altura)
    px2 = int((det['x_centro'] + det['largura'] / 2) * largura)
    py2 = int((det['y_centro'] + det['altura']  / 2) * altura)

    cor   = (0, 0, 255) if det['classe'] == 1 else (0, 255, 0)   # BGR
    label = 'SEM CAPACETE' if det['classe'] == 1 else 'com capacete'

    cv2.rectangle(imagem_para_desenhar, (px1, py1), (px2, py2), cor, thickness=3)
    cv2.putText(imagem_para_desenhar, label, (px1, max(py1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor, thickness=2)

imagem_final = cv2.cvtColor(imagem_para_desenhar, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(9, 6))
plt.imshow(imagem_final)
plt.title('Etapa 3 — Frame com bounding box desenhada pelo OpenCV')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Etapa 4 — Salvar o resultado final
*(Aula 4.1 — imwrite)*

In [ ]:
# ── 6. Salvar o frame processado (precisa estar em BGR para salvar correto) ──
cv2.imwrite('frame_processado.jpg', imagem_para_desenhar)
print("✅ Resultado salvo como 'frame_processado.jpg'")

# ── 7. Painel final — resumo visual de todo o pipeline ────────────────────────
fig, axs = plt.subplots(1, 4, figsize=(18, 5))

axs[0].imshow(imagem_rgb)
axs[0].set_title('1. Original')
axs[0].axis('off')

axs[1].imshow(roi)
axs[1].set_title('2. Recorte (ROI)')
axs[1].axis('off')

axs[2].imshow(imagem_cinza, cmap='gray')
axs[2].set_title('3. Escala de cinza')
axs[2].axis('off')

axs[3].imshow(imagem_final)
axs[3].set_title('4. Com bounding box')
axs[3].axis('off')

fig.suptitle('Pipeline completo do Bloco 4 — OpenCV', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

---
## Resumo do Projeto Integrador

<table style='border-collapse: collapse; width: 100%; font-size: 15px;'>
  <thead>
    <tr style='background-color: #2c2c2c; color: white;'>
      <th style='padding: 10px; border: 1px solid #555;'>Etapa</th>
      <th style='padding: 10px; border: 1px solid #555; background-color: #e07b00;'>Função no código</th>
      <th style='padding: 10px; border: 1px solid #555;'>O que faz</th>
    </tr>
  </thead>
  <tbody>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>1. Carregar e corrigir cor</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.imread()</code> + <code>cvtColor(BGR2RGB)</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Abrir o frame do drone corretamente</td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>2. Recorte e escala de cinza</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>imagem[y1:y2, x1:x2]</code> + <code>COLOR_BGR2GRAY</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Isolar regiões e reduzir dados</td>
    </tr>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>3. Desenhar detecção</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.rectangle()</code> + <code>cv2.putText()</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Marcar e rotular o resultado do YOLO</td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>4. Salvar resultado</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.imwrite()</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Guardar o frame processado</td>
    </tr>
  </tbody>
</table>

### Conclusão do Bloco 4

Você agora sabe o **pipeline completo de processamento de imagem** usado antes e depois do YOLO:
carregar corretamente, manipular (recortar/converter) e desenhar os resultados das detecções.

**Próximo passo → Bloco 5: YOLO** — treinar o modelo de verdade nas imagens do drone 🚁